In [ ]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


# Overview dashboard

High-level view of the 287 Claude Code system prompts. Three charts:

1. **Linked dashboard** (per-file scatter ↔ per-category bars) — brush a region in the scatter to filter the bars on the right.
2. **Modality grouped bars** (deontic / epistemic / dynamic per category).
3. **Findings markdown** — headline patterns, including the absence-as-signal observations for `collaborative` and `appreciative` sentence-register classes.

Run **`00_data_pipeline.ipynb`** first to produce `prompt_linguistic_analysis.yaml`. The setup cell below loads from that YAML; nothing in this notebook re-runs spaCy.


### Linked dashboard: per-file scatter ↔ category averages

Brush a region in the scatter on the left to filter the bar charts on the right. Hover any point
for the file path and exact metric values. Click a category in the legend to highlight only that
category.

In [ ]:
"""Linked dashboard: file-level scatter + brush-filtered category aggregates.

Tooltips show ``name`` and ``description`` from the HTML-comment frontmatter — much
more readable than the raw filename."""

# Tableau 10 mapped to the categories present in the corpus (matches matplotlib chart 33).
_cat_domain = list(by_category.keys())
_cat_range  = [CATEGORY_COLORS[c] for c in _cat_domain]
cat_color = alt.Color("category:N",
                       scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                       legend=alt.Legend(title="Category", orient="bottom", columns=4))

brush = alt.selection_interval(encodings=["x", "y"])
legend_sel = alt.selection_point(fields=["category"], bind="legend")

scatter = (
    alt.Chart(alt_df)
    .mark_circle(opacity=0.7)
    .encode(
        x=alt.X("mood_marker_pct:Q",
                title="Imperative-marker density (% of file tokens)"),
        y=alt.Y("just_ratio:Q", title="Justification ratio (reasons / imperative)"),
        size=alt.Size("n_tokens:Q",
                      title="tokens",
                      scale=alt.Scale(range=[20, 600])),
        color=cat_color,
        opacity=alt.condition(legend_sel, alt.value(0.85), alt.value(0.07)),
        tooltip=[
            alt.Tooltip("name:N",        title="Name"),
            alt.Tooltip("description:N", title="Description"),
            alt.Tooltip("ccVersion:N",   title="ccVersion"),
            alt.Tooltip("path:N",        title="File"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q",    format=","),
            alt.Tooltip("imperative_sent_pct:Q",       title="imperative % of sents", format=".1f"),
            alt.Tooltip("mood_marker_pct:Q",           title="imp markers %",         format=".2f"),
            alt.Tooltip("just_ratio:Q",                title="justification ratio",   format=".2f"),
            alt.Tooltip("hard_prohibitions_pct:Q",     title="hard_proh %",           format=".2f"),
            alt.Tooltip("caps_imp_pct:Q",              title="CAPS imp %",            format=".2f"),
            alt.Tooltip("dominant_stance:N"),
            alt.Tooltip("dominant_register:N"),
            alt.Tooltip("dominant:N",                  title="dominant sentence-register"),
        ],
    )
    .add_params(brush, legend_sel)
    .properties(width=470, height=420,
                title="Per-file: imperative density vs justification ratio (brush to filter →)")
)

metrics = [
    ("hard_prohibitions_pct", "Hard prohibitions %"),
    ("caps_imp_pct",          "CAPS imperative %"),
    ("all_caps_pct",          "ALL CAPS %"),
]

linked_bars = []
for col, title in metrics:
    bar = (
        alt.Chart(alt_df)
        .mark_bar()
        .encode(
            x=alt.X(f"mean({col}):Q", title=f"{title} (of file tokens)"),
            y=alt.Y("category:N", sort="-x", title=None),
            color=cat_color,
            tooltip=[
                alt.Tooltip("category:N"),
                alt.Tooltip(f"mean({col}):Q", title=f"mean {title}", format=".3f"),
                alt.Tooltip("count():Q", title="files in selection"),
            ],
        )
        .transform_filter(brush)
        .properties(width=260, height=130, title=f"{title} (mean, in selection)")
    )
    linked_bars.append(bar)

dashboard = scatter | alt.vconcat(*linked_bars)
dashboard


### Modality per category (single-source spaCy detector)

Single-panel grouped bar showing deontic / epistemic / dynamic density per category,
in % of file tokens. Hover any bar for the underlying value; click a category name in
the legend to highlight just that category.

In [ ]:
"""Modality per category × class (single-source). Bars in % of file tokens."""

mod_long = pd.DataFrame([
    {"category": cat, "class": cls,
     "value": by_category[cat]["metrics"]["modality"][f"{cls}_pct"]}
    for cat in cats
    for cls in ("deontic", "epistemic", "dynamic")
])

cat_sel = alt.selection_point(fields=["category"], bind="legend")

mod_chart = (
    alt.Chart(mod_long)
    .mark_bar()
    .encode(
        x=alt.X("class:N", title=None,
                sort=["deontic", "epistemic", "dynamic"]),
        y=alt.Y("value:Q", title="% of file tokens"),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=_cat_domain, range=_cat_range)),
        opacity=alt.condition(cat_sel, alt.value(0.95), alt.value(0.1)),
        column=alt.Column("category:N", title=None),
        tooltip=[
            alt.Tooltip("category:N"),
            alt.Tooltip("class:N"),
            alt.Tooltip("value:Q", format=".3f", title="% of file tokens"),
        ],
    )
    .add_params(cat_sel)
    .properties(width=110, height=240,
                title="Modality per category (% of file tokens)")
)
mod_chart


## 13. Findings

Headline patterns from this analysis (densities are **% of file tokens** unless stated otherwise; sentence-register figures are **% of sentences**).

- **The corpus is overwhelmingly directive prose, not conversation.** Across all 287 files, ~31 % of *sentences* are imperative (per `metrics.sentence_register.imperative_sent_pct`), ~14 % carry a directive marker, and only ~6 % concern configuration. The lexicon confirms: **628** hard-prohibitions vs **0** profanity hits; **1387** second-person ("you") pronouns vs **153** first-person ("we/I") — a **9:1** ratio.

- **Absence is the signal — what the corpus is NOT.** The new sentence-level pragmatic register classifier surfaces three **near-zero** classes corpus-wide:
  - `collaborative_sent_pct` ≈ **0.5 %** (29 sentences out of 5,694 — interpersonal "let's", "we should", "together").
  - `appreciative_sent_pct` ≈ **0.07 %** (just **4 sentences** out of 5,694 — gratitude, praise, thanks).
  - `permissive_sent_pct` ≈ **2.2 %** (126 sentences — "please", "you can", "you may").
  Claude Code system prompts are functional/directive, not interpersonal. Knowing what the corpus *isn't* doing is as important as knowing what it is — these classes are kept deliberately so the absence is measurable.

- **Tool descriptions are the most prohibition-heavy category.** They have the highest density of `hard_prohibitions` (~1.18 % of tokens, ≈ 1 prohibition every 85 tokens) and the highest density of imperative markers. The most extreme outlier files are the bash-sandbox tool descriptions (`bash-sandbox-no-exceptions.md`, `bash-sandbox-evidence-operation-not-permitted.md`), which run at 9 % hard-prohibitions per token — about one prohibition every 11 tokens.

- **System reminders are the most CAPS-heavy.** ALL CAPS tokens make up ~7.6 % of their tokens (≈ 4× any other category). They're terse, declarative-leaning, and emphatic.

- **Stance polarity is heavily positive.** With `evaluative` split into `positive_evaluative` and `negative_evaluative`, the corpus shows ~3–6× more positive than negative judgment markers across nearly every category. The signal is "this is the right way", not "that was the wrong way".

- **Skills and Agent prompts lead on justification ratio** (~0.41 and 0.36). These prompts pair rules with stated reasons. Tool descriptions (0.15) and System reminders (0.10) issue rules without explanations. The single file with the highest ratio is `agent-prompt-agent-creation-architect.md` at 3.0.

- **Modality (single-source spaCy detector) — what dominates per category:**
  - *Tool description* leads on deontic (0.50 % of tokens) and dynamic (0.65 %) — instructional "must / can do this" language with light epistemic content.
  - *System reminder* has the highest deontic density (0.88 %) — short emphatic obligation prompts.
  - *Skill* and *Data / template* are modality-light overall — they describe rather than command.
  - The top corpus-wide modal construction is `can` (post-audit; previously `should be`).

- **Stance fingerprint per category:**
  - *Tool description* and *System prompt* lean **directive** (highest density of `must` / `should` / `do not` / `you must`).
  - *Skill*, *Data / template*, and *Agent prompt* lean **expository** (`is a`, `means`, `for example`).
  - *System reminder* mixes directive and expository, with the highest dialogic density (`but` / `however` / `though`) — they often inject a contrastive note.

- **CAPS-imperative tokens cluster around safety-critical content.** Top emphatic tokens corpus-wide: `IMPORTANT (37)`, `NEVER (27)`, `MUST (21)`, `CRITICAL (17)`, `DO NOT (15)`. The files with the highest CAPS-imperative density are bash-sandbox descriptions, malware-analysis reminders, and chrome-browser MCP system prompts — exactly the contexts where Claude must not drift.

The full per-file numbers (one row per `.md`) and the lexicons used live in `prompt_linguistic_analysis.yaml` next to this notebook.
